In [88]:
import os
from neo4j import GraphDatabase
from yfiles_jupyter_graphs import GraphWidget
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough, ConfigurableField
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.prompt import PromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field
from typing import List
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.graphs.neo4j_graph import Neo4jGraph
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_community.vectorstores import Neo4jVector
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores.neo4j_vector import remove_lucene_chars
from langchain.llms import HuggingFaceHub
from tqdm import tqdm
from multiprocessing import Pool
from modules import logging

In [2]:
# import multiprocessing

# cpu_cores = multiprocessing.cpu_count()
# print(f"전체 CPU 코어 개수: {cpu_cores}")

In [41]:
logging.langsmith("PDF-RAG")

LangSmith 추적을 시작합니다.
[프로젝트명]
PDF-RAG


In [42]:
# API KEY를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API KEY 정보로드
load_dotenv()

True

In [6]:
def show_metadata(docs):
    if docs:
        print("[metadata]")
        print(list(docs[0].metadata.keys()))
        print("\n[examples]")
        max_key_length = max(len(k) for k in docs[0].metadata.keys())
        for k, v in docs[0].metadata.items():
            print(f"{k:<{max_key_length}} : {v}")

In [71]:
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass(prompt = 'OpenAI')


In [84]:
os.environ["NEO4J_URI"] = getpass.getpass(prompt = 'neo4j+s://db8bf133.databases.neo4j.io')
os.environ["NEO4J_USERNAME"] = getpass.getpass(prompt = 'neo4j')
os.environ["NEO4J_PASSWORD"] = getpass.getpass(prompt = 'gTCqLY75bwR5rz5Fc8127o8DFkwkmfErDgIg_-ndiPM')
# os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass.getpass(prompt = 'HUGGINGFACEHUB_API_TOKEN')

In [47]:

FILE_PATH = "data\메리츠_1-3.pdf"

In [48]:
raw_documents = PyPDFLoader(FILE_PATH).load()

In [55]:
print(raw_documents[2].page_content)

해당 중도인출금을 차감합니다.⑤ 보험금 지급사유가 회사의 자체적인 기준이 아닌 계약에 관한 사항   가. 다른 법률과 보험금 지급사유가 연계되는 등 보험금 지급사유가 회사의 자체적인 기준이 아님에 따라 아래와 같은 경우가 발생되는 경우 회사는 객관적이고 합리적인 범위 내에서 기존 계약내용에 상응하는 새로운 보장내용으로 계약내용을 변경할 수 있습니다.       1) 관련 법률의 개정 또는 폐지 등에 따라 약관에서 정한 보험금 지급사유 판정기준이 변경되는 경우       2) 관련 법률의 개정 또는 폐지 등에 따라 약관에서 정한 보험금 지급사유의 판정이 불가능한 경우       3) 관련 법률의 개정 또는 폐지 등에 따라 계약유지 필요가 없어지는 경우       4) 기타 금융위원회 등의 명령이 있는 경우   나. 회사는 가.에 따라 계약이 변경되는 경우 계약내용 변경일의 15일 이전까지 서면(등기우편 등), 전화(음성녹취) 또는 전자문서 등으로 보장내용, 가입금액, 보험료 변경내용 및 변경 절차 등을 계약자에게 알립니다.   다. 가.에 따라 계약내용을 변경하는 경우 보장내용, 가입금액 및 납입보험료가 변경될 수 있으며, 계약내용 변경 시점 이후 잔여보험기간의 보장을 위한 재원인 계약자적립액 및 미경과보험료 정산으로 계약자가 추가로 납입하여야 할 (또는 반환받을)금액이 발생할 수 있으며, 이를 계약 체결시 계약자에게 안내합니다.   라. 회사는 가.에 따라 보장내용이 변경되는 경우 최신의 통계를 반영하여 보험료산출기초율을 재산출할 수 있으며 다음과 같이 적용합니다.      : 계약내용 변경일부터 재산출된 보험료산출기초율을 적용할 수 있으며, 이미 체결한 계약에 대하여 보험료 또는 보험금이 변경될 수 있습니다.   마. 가.에도 불구하고 계약자가 계약내용 변경을 원하지 않거나 새로운 보장내용으로 계약내용을 변경하는 것이 불가능한 경우 회사는 계약자에게「보험료 및 해약환급금 산출방법서」에서 정하는 바에 따라 계약내용 변경시점의 계약자적립액 및 미경과보험료를 지급

In [54]:
print(f"컨텐츠 글자의 수: {len(raw_documents[1].page_content)}")

컨텐츠 글자의 수: 1418


In [11]:
show_metadata(raw_documents)

[metadata]
['source', 'page']

[examples]
source : data\무배당 메리츠 다이렉트 운전자보험2409.pdf
page   : 0


In [56]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=125)

In [57]:
documents = text_splitter.split_documents(raw_documents)

In [58]:
print(f"분할된 청크의수: {len(documents)}")

분할된 청크의수: 12


In [15]:
# repo_id = 'Bllossom/llama-3.2-Korean-Bllossom-3B'

In [79]:
llm = ChatOpenAI(model_name="gpt-3.5-turbo-0125", temperature=0)

In [16]:
# llm = HuggingFaceHub(
#     repo_id=repo_id,
#     model_kwargs={"temperature": 0, # 자유도
#                   "max_tokens":150} # 최대 답변 길이
# )

C:\Users\vkxql\AppData\Local\Temp\ipykernel_18196\994402920.py:1: LangChainDeprecationWarning: The class `HuggingFaceHub` was deprecated in LangChain 0.0.21 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEndpoint``.
  llm = HuggingFaceHub(


In [80]:
llm_transformer = LLMGraphTransformer(llm=llm)

In [81]:
# 예시: convert_to_graph_documents가 반복 가능한 객체를 처리하는 경우
def convert_to_graph_documents_with_progress(documents):
    processed_documents = []
    
    # TQDM 진행바 추가
    for doc in tqdm(documents, desc="Converting documents to graph format"):
        # 실제 변환 작업 수행
        processed_doc = llm_transformer.convert_to_graph_documents([doc])  # 단일 문서 처리
        processed_documents.append(processed_doc)
    
    return processed_documents

In [82]:
graph_documents = convert_to_graph_documents_with_progress(documents)

Converting documents to graph format: 100%|██████████| 12/12 [01:29<00:00,  7.45s/it]


In [89]:
graph = Neo4jGraph()

Unable to retrieve routing information


ValueError: Could not connect to Neo4j database. Please ensure that the url is correct